# Allen CT — Population Patch Analysis

Across-cell analysis of spikeparam waveform features.

**Grouping variables:**
- `dendrite_type` (spiny = PC, aspiny = IN)
- `stimulus_name` (Long Square, Short Square, Ramp, Noise, …)

**Tables (relational, loaded below):**
- `allen_ct_population_spikes.pkl` — one row per spike
- `allen_ct_sweep_table.pkl`       — one row per spiking sweep
- `load_cell_metadata()`           — one row per cell

In [ ]:
%load_ext autoreload
%autoreload 3
%matplotlib inline
%config InlineBackend.figure_format = 'retina'

import sys, os, pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# ── adjust these two paths if opening from a different working directory ──────
_here = os.path.dirname(os.path.abspath('__file__'))
sys.path.insert(0, os.path.join(_here, '../../allen_ct_helper_modules'))
sys.path.insert(0, os.path.join(_here, '../../../..'))

from data_loader import load_cell_metadata, get_all_spiking_sweeps, load_voltage_trace
from config import ALLEN_CT_PICKLE_ROOT
from spikeparam.patch.fit import Spike

import warnings
warnings.filterwarnings('ignore')

COL_SPINY  = '#CC44CC'
COL_ASPINY = '#00CCCC'
COL_SPARSE = '#AAAAAA'
DEND_COLS  = {'spiny': COL_SPINY, 'aspiny': COL_ASPINY, 'sparsely spiny': COL_SPARSE}
STIM_COLS  = {
    'Long Square':                '#2166AC',
    'Short Square':               '#D53E4F',
    'Ramp':                       '#F46D43',
    'Noise 1':                    '#66C2A5',
    'Noise 2':                    '#3288BD',
    'Square - 2s Suprathreshold': '#ABDDA4',
    'Short Square - Triple':      '#FDAE61',
}
FS_TITLE, FS_LABEL, FS_TICK, FS_ANNOT = 11, 10, 9, 8
FEAT_DIR = os.path.join(ALLEN_CT_PICKLE_ROOT, 'features')
SPIKE_FEAT_COLS = [
    'ramp_amp', 'inflection_time', 'inflection_amp',
    'peak_amp', 'peak_width', 'peak_sharpness',
    'exp_lambda', 'exp_const', 'log_isi',
]
FEAT_LABELS = {
    'ramp_amp':        'Ramp amp (mV/ms)',
    'inflection_time': 'Threshold time (ms)',
    'inflection_amp':  'Threshold voltage (mV)',
    'peak_amp':        'Peak amplitude (mV)',
    'peak_width':      'Spike width (ms)',
    'peak_sharpness':  'Peak sharpness',
    'exp_lambda':      'Repolarization rate λ',
    'exp_const':       'Repolarization const (mV)',
    'log_isi':         'Log ISI',
}

In [ ]:
spikes   = pd.read_pickle(os.path.join(ALLEN_CT_PICKLE_ROOT, 'allen_ct_population_spikes.pkl'))
sweeps   = pd.read_pickle(os.path.join(ALLEN_CT_PICKLE_ROOT, 'allen_ct_sweep_table.pkl'))
cells_df = load_cell_metadata(species='Mus musculus').rename(columns={'id': 'specimen_id'})

SWEEP_COLS = [c for c in ['sweep_uid','stimulus_name','stimulus_absolute_amplitude',
                           'num_spikes','pre_vm_mv'] if c in sweeps.columns]
CELL_COLS  = [c for c in ['specimen_id','dendrite_type','structure_area_abbrev',
                           'structure_layer_name','transgenic_line','normalized_depth']
              if c in cells_df.columns]

full_df = (
    spikes
    .merge(sweeps[SWEEP_COLS], on='sweep_uid', how='left')
    .merge(cells_df[CELL_COLS],  on='specimen_id', how='left')
)
FEAT_COLS = [c for c in SPIKE_FEAT_COLS if c in full_df.columns]

print(f'Spikes:  {len(full_df):,}')
print(f'Cells:   {full_df["specimen_id"].nunique()}')
print(f'Sweeps:  {full_df["sweep_uid"].nunique()}')
print()
print('── By dendrite type ─────')
print(full_df.groupby('dendrite_type')['specimen_id'].agg(spikes='count', cells='nunique'))
print()
print('── By stimulus type (top 8) ─────')
print(full_df.groupby('stimulus_name').size().sort_values(ascending=False).head(8))

## 1. Mean waveforms by cell type

Re-fit Spike on the best Long Square sweep for a sample of spiny and aspiny cells, then overlay mean waveforms.

In [ ]:
def _best_long_square(sid):
    sweeps_cell = get_all_spiking_sweeps(sid)
    ls = [s for s in sweeps_cell if 'Long Square' in s.get('stimulus_name', '')]
    s  = max(ls or sweeps_cell, key=lambda x: x.get('num_spikes') or 0)
    v, _, _, fs, idx = load_voltage_trace(sid, s)
    if idx is not None:
        v = v[idx[0]:idx[1]]
    sp = Spike(thresh_amp=-10, window_length=(5., 5.), smooth_frac=0.008, pre_inflection_ms=0.5)
    sp.fit(list(v), fs, n_jobs=-1)
    sp.filter_features()
    return sp

processed_ids = {
    int(f.replace('_features.pkl',''))
    for f in os.listdir(FEAT_DIR) if f.endswith('_features.pkl')
}

N_SAMPLE = 3   # cells per type to overlay (increase after full batch)
mean_wavs = {}
for dt, c in [('spiny', COL_SPINY), ('aspiny', COL_ASPINY)]:
    cands = cells_df[
        (cells_df['dendrite_type'] == dt) &
        (cells_df['specimen_id'].isin(processed_ids))
    ]['specimen_id'].tolist()[:N_SAMPLE]
    mean_wavs[dt] = []
    for sid in cands:
        try:
            sp = _best_long_square(int(sid))
            mean_wavs[dt].append(np.array(sp.waveforms).mean(axis=0))
        except Exception as e:
            print(f'  skipped {sid}: {e}')

# Plot
fig, ax = plt.subplots(figsize=(8, 5))
for dt, c in [('spiny', COL_SPINY), ('aspiny', COL_ASPINY)]:
    for i, wm in enumerate(mean_wavs[dt]):
        t_ms = np.linspace(-5, 5, len(wm))
        ax.plot(t_ms, wm, color=c, alpha=0.5 if i > 0 else 1.0,
                linewidth=1.5 if i > 0 else 2.5,
                label=f'{dt} (n={len(mean_wavs[dt])})' if i == 0 else None)
ax.axvline(0, color='gray', linestyle='--', linewidth=0.8, alpha=0.6)
ax.set_xlabel('Time from peak (ms)', fontsize=FS_LABEL)
ax.set_ylabel('Voltage (mV)', fontsize=FS_LABEL)
ax.set_title('Mean spike waveforms — spiny (PC) vs aspiny (IN)',
             fontsize=FS_TITLE, fontweight='bold', loc='left')
ax.legend(fontsize=FS_ANNOT, frameon=False)
sns.despine(ax=ax)
plt.tight_layout()
plt.show()

## 2. Feature distributions by cell type

Violin + strip for all spikeparam features, split by dendrite type.

In [ ]:
def plot_by_dend(df, title_suffix=''):
    DEND_ORDER = ['spiny', 'aspiny']
    DEND_C     = [COL_SPINY, COL_ASPINY]
    sub = df[df['dendrite_type'].isin(DEND_ORDER)]
    n_cols = 3
    n_rows = int(np.ceil(len(FEAT_COLS) / n_cols))
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(4*n_cols, 3.5*n_rows))
    axes_flat = list(axes.flat) if n_rows > 1 else list(axes)
    rng = np.random.default_rng(42)
    for ax, feat in zip(axes_flat, FEAT_COLS):
        for i, (dt, c) in enumerate(zip(DEND_ORDER, DEND_C)):
            vals = sub[sub['dendrite_type'] == dt][feat].dropna().values
            if len(vals) < 5:
                continue
            parts = ax.violinplot(vals, positions=[i], widths=0.65,
                                  showmedians=False, showextrema=False)
            for pc in parts['bodies']:
                pc.set_facecolor(c); pc.set_alpha(0.35); pc.set_linewidth(0.5)
            jit = rng.uniform(-0.12, 0.12, len(vals))
            ax.scatter(i + jit, vals, color=c, s=3, alpha=0.5, linewidths=0, zorder=3)
            ax.plot([i-.18, i+.18], [np.median(vals)]*2, 'k-', lw=2.2, zorder=5)
        n_sp = sub[sub['dendrite_type']=='spiny'][feat].dropna().shape[0]
        n_as = sub[sub['dendrite_type']=='aspiny'][feat].dropna().shape[0]
        ax.set_xticks([0,1])
        ax.set_xticklabels([f'spiny\n(PC)\nn={n_sp}', f'aspiny\n(IN)\nn={n_as}'],
                           fontsize=FS_TICK)
        ax.set_ylabel(FEAT_LABELS.get(feat, feat), fontsize=FS_LABEL)
        ax.set_title(FEAT_LABELS.get(feat, feat), fontsize=FS_TITLE, fontweight='bold', loc='left')
        ax.set_xlim(-0.6, 1.6)
        sns.despine(ax=ax)
    for ax in axes_flat[len(FEAT_COLS):]:
        ax.set_visible(False)
    n_sp_c = sub[sub['dendrite_type']=='spiny']['specimen_id'].nunique()
    n_as_c = sub[sub['dendrite_type']=='aspiny']['specimen_id'].nunique()
    fig.suptitle(f'Features by dendrite type  |  {title_suffix}'
                 f'  |  spiny n={n_sp_c} cells  aspiny n={n_as_c} cells',
                 fontsize=12, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.show()

plot_by_dend(full_df, 'all stimuli')

In [ ]:
ls_df = full_df[full_df['stimulus_name'].str.contains('Long Square', na=False)]
plot_by_dend(ls_df, 'Long Square only')

## 3. Feature distributions by stimulus type

In [ ]:
top_stims   = (full_df.groupby('stimulus_name').size()
               .sort_values(ascending=False).head(5).index.tolist())
stim_colors = [STIM_COLS.get(s, '#888888') for s in top_stims]
stim_sub    = full_df[full_df['stimulus_name'].isin(top_stims)]

n_cols = 3
n_rows = int(np.ceil(len(FEAT_COLS) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(4*n_cols, 3.5*n_rows))
axes_flat = list(axes.flat) if n_rows > 1 else list(axes)
rng2 = np.random.default_rng(7)
for ax, feat in zip(axes_flat, FEAT_COLS):
    for i, (stim, c) in enumerate(zip(top_stims, stim_colors)):
        vals = stim_sub[stim_sub['stimulus_name'] == stim][feat].dropna().values
        if len(vals) < 5:
            continue
        parts = ax.violinplot(vals, positions=[i], widths=0.7,
                              showmedians=False, showextrema=False)
        for pc in parts['bodies']:
            pc.set_facecolor(c); pc.set_alpha(0.4); pc.set_linewidth(0.5)
        jit = rng2.uniform(-0.12, 0.12, len(vals))
        ax.scatter(i + jit, vals, color=c, s=3, alpha=0.5, linewidths=0, zorder=3)
        ax.plot([i-.2, i+.2], [np.median(vals)]*2, 'k-', lw=2, zorder=5)
    ax.set_xticks(range(len(top_stims)))
    ax.set_xticklabels([s.replace(' ','\n') for s in top_stims], fontsize=6)
    ax.set_ylabel(FEAT_LABELS.get(feat, feat), fontsize=FS_LABEL)
    ax.set_title(FEAT_LABELS.get(feat, feat), fontsize=FS_TITLE, fontweight='bold', loc='left')
    ax.set_xlim(-0.6, len(top_stims) - 0.4)
    sns.despine(ax=ax)
for ax in axes_flat[len(FEAT_COLS):]:
    ax.set_visible(False)
fig.suptitle('Features by stimulus type  |  top 5 stimulus types by spike count',
             fontsize=12, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## 4. Feature correlation structure

Spearman correlation heatmap — spiny vs aspiny separately.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, dt, c in zip(axes, ['spiny', 'aspiny'], [COL_SPINY, COL_ASPINY]):
    sub  = full_df[full_df['dendrite_type'] == dt][FEAT_COLS].dropna()
    corr = sub.corr(method='spearman')
    labels = [FEAT_LABELS.get(f, f) for f in FEAT_COLS]
    sns.heatmap(corr, ax=ax, annot=True, fmt='.2f', annot_kws={'size': 7},
                cmap='RdBu_r', vmin=-1, vmax=1, linewidths=0.4,
                xticklabels=labels, yticklabels=labels,
                cbar_kws={'shrink': 0.7})
    n_c = full_df[full_df['dendrite_type'] == dt]['specimen_id'].nunique()
    ax.set_title(f'{dt}  |  n={n_c} cells  {len(sub):,} spikes',
                 fontsize=FS_TITLE, fontweight='bold', loc='left')
    ax.tick_params(axis='x', rotation=45, labelsize=7)
    ax.tick_params(axis='y', rotation=0,  labelsize=7)
fig.suptitle('Spearman feature correlations by dendrite type', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('allen_ct_feature_correlations.pdf', bbox_inches='tight', dpi=300)
plt.savefig('allen_ct_feature_correlations.png', bbox_inches='tight', dpi=300)
plt.show()